# WikiGraph Agent Demo
## LLM Wiki Philosophy + LightRAG Engine

LLM Wiki의 "지식이 자라는" 철학을 LightRAG의 그래프 엔진 위에서 실현합니다.

1. **INGEST** — 문서 삽입 + 검증
2. **QUERY** — 그래프 즉시 검색 + 쿼리 로그
3. **EVOLVE** — 쿼리 패턴 분석 → 지식 자동 진화
4. **LINT** — 그래프 건강검진

## 1. Setup

In [ ]:
import sys, os, shutil
import numpy as np
sys.path.insert(0, '..')
sys.path.insert(0, '.')

from sentence_transformers import SentenceTransformer
from lightrag import LightRAG, QueryParam
from lightrag.llm.openai import openai_complete_if_cache
from lightrag.utils import EmbeddingFunc

LLM_BASE_URL = 'http://222.117.133.162:30010/v1'
LLM_MODEL    = 'qwen-task-pool'
LLM_API_KEY  = 'asdf'
EMBED_MODEL  = 'sentence-transformers/all-MiniLM-L6-v2'
EMBED_DIM    = 384
WORK_DIR     = '/tmp/wikigraph_demo'

print(f'Loading embedding model: {EMBED_MODEL} ...')
embed_model = SentenceTransformer(EMBED_MODEL)
print('Done.')

async def llm_func(prompt, system_prompt=None, history_messages=[], **kwargs):
    return await openai_complete_if_cache(
        LLM_MODEL, prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        api_key=LLM_API_KEY,
        base_url=LLM_BASE_URL,
        **kwargs,
    )

async def embed_func(texts):
    return embed_model.encode(texts, normalize_embeddings=True)

print(f'LLM: {LLM_BASE_URL} ({LLM_MODEL})')
print(f'Embedding: {EMBED_MODEL} (local)')

## 2. LightRAG + WikiGraph Agent 초기화

In [ ]:
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

rag = LightRAG(
    working_dir=WORK_DIR,
    llm_model_func=llm_func,
    embedding_func=EmbeddingFunc(
        embedding_dim=EMBED_DIM,
        max_token_size=8192,
        func=embed_func,
    ),
    addon_params={
        'enable_hybrid_search': True,
        'hybrid_search_mode': 'hybrid',
    },
)
await rag.initialize_storages()

from wikigraph.agent import WikiGraphAgent
from wikigraph.config import WikiGraphConfig

config = WikiGraphConfig(working_dir=WORK_DIR)
agent = WikiGraphAgent(rag, config, llm_func=llm_func)

print(f'LightRAG ready: {WORK_DIR}')
print(f'WikiGraph Agent initialized')
print(f'Hybrid Search: {rag._addon_params["enable_hybrid_search"]}')

## 3. INGEST — 문서 삽입

LLM Wiki처럼 문서를 넣으면 자동으로:
- 청킹 → 임베딩 → 엔티티/관계 추출 → 그래프 저장 → BM25 인덱싱

LLM Wiki와 다른 점: **증분 업데이트** — 기존 그래프를 재구축하지 않음

In [ ]:
# 작은 문서 2개로 테스트
docs_dir = os.path.join('..', 'docs')
doc_files = ['FrontendBuildGuide.md', 'UV_LOCK_GUIDE.md']

documents = []
for fname in doc_files:
    with open(os.path.join(docs_dir, fname), 'r') as f:
        documents.append(f.read())
    print(f'  {fname}: {len(documents[-1]):,} chars')

print(f'\nIngesting {len(documents)} documents...')
result = await agent.ingest(documents, file_paths=doc_files)
for msg in result['messages']:
    print(f'  {msg}')

## 4. QUERY — 쿼리 + 자동 평가

쿼리 결과를 자동 평가하고, 품질이 낮으면 EVOLVE를 트리거합니다.
쿼리 로그가 축적되어 나중에 EVOLVE의 입력이 됩니다.

In [ ]:
queries = [
    'How to build the frontend WebUI?',
    'What is UV lock file used for?',
    'How to install frontend dependencies?',
    'What build tools does LightRAG use?',
    'How does bun relate to the frontend build process?',
]

for q in queries:
    print(f'\n{"="*60}')
    result = await agent.query(q)
    for msg in result['messages'][-3:]:
        print(f'  {msg}')
    if result.get('evolved'):
        print('  >>> Knowledge evolved!')

print(f'\n--- Agent Stats ---')
stats = agent.stats
print(f'Total queries: {stats["total_queries"]}')
print(f'Tracked entities: {stats["tracked_entities"]}')

## 5. EVOLVE — 지식 진화

쿼리 로그를 분석하여:
1. 자주 함께 검색되는 엔티티 → 새 관계 생성
2. 실패한 쿼리 → 지식 갭 채우기
3. 멀티홉 경로 → 단축 관계 생성

In [ ]:
print('Running EVOLVE...')
result = await agent.evolve()
for msg in result['messages'][-10:]:
    print(f'  {msg}')
print(f'\nMutations applied: {result["applied"]}')

## 6. LINT — 그래프 건강검진

LLM Wiki는 전체 wiki를 LLM으로 스캔해야 하지만,
WikiGraph는 **그래프 알고리즘**으로 0토큰 탐지합니다.

In [ ]:
print('Running LINT...')
result = await agent.lint()
for msg in result['messages'][-10:]:
    print(f'  {msg}')

if result['findings']:
    print(f'\n--- Findings ({len(result["findings"])}) ---')
    for f in result['findings'][:10]:
        print(f'  [{f["severity"]}] {f["finding_type"]}: {f["entity_name"]}')
        print(f'    {f["details"]}')
        print(f'    Action: {f["suggested_action"]}')

## 7. 그래프 시각화

In [ ]:
import networkx as nx

G = nx.read_graphml(os.path.join(WORK_DIR, 'graph_chunk_entity_relation.graphml'))
print(f'Knowledge Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

# Show evolved edges (source_id contains 'wikigraph_evolve')
evolved_edges = [(u, v) for u, v, d in G.edges(data=True) if 'wikigraph_evolve' in d.get('source_id', '')]
print(f'Evolved edges: {len(evolved_edges)}')
for u, v in evolved_edges:
    print(f'  {u} → {v} (auto-generated by EVOLVE)')

print(f'\nTop entities by degree:')
degrees = sorted(G.degree(), key=lambda x: x[1], reverse=True)
for name, deg in degrees[:10]:
    print(f'  {name:30s} degree={deg}')

## 8. Cleanup

In [ ]:
await rag.finalize_storages()
print('Done.')

## Summary

| 차원 | LLM Wiki v2 | WikiGraph Agent |
|---|---|---|
| **스케일** | ~1000페이지 | 수만 노드 |
| **업데이트** | 페이지 15개 재작성 | 노드/엣지 증분 |
| **검색** | 풀컨텍스트 로딩 | 그래프+벡터+BM25 |
| **오류** | 전파됨 | 원본 보존+검증 |
| **건강검진** | LLM 전체 스캔 | 그래프 알고리즘 |
| **지식 진화** | LLM 재작성 | 쿼리 패턴 기반 자동 |

```
WikiGraph = LLM Wiki의 철학 + LightRAG의 엔진
         = 스케일러블하게 자라는 지식 그래프
```